In [1]:
import torchdiffeq
import dill as pickle
import torch
import numpy as np
from data_process import DataProcessor
import os
import time
import torch.nn as nn
import torch.optim as optim

In [2]:
robots_form = 'circle' 
if robots_form == 'circle':
    with open('circle/data/circle_data_00_330_[30_bots_PWM_10_15cw_15ccw_D_41cm].MP4.pickle', 'rb') as file: 
        raw_data = pickle.load(file)

In [3]:
processor = DataProcessor(raw_data)
coord_data, angle_data = processor.extract_data()
normalized_coord_data = processor.normalize_all_coordinates()

In [4]:
normalized_coords = np.array([normalized_coord_data[r] for r in normalized_coord_data])

x = normalized_coords[25:26,:200,0]
y = normalized_coords[25:26,:200,1]

vx = np.gradient(x, axis=1)
vy = np.gradient(y, axis=1)

state = np.stack(
    [x,y, vx, vy],
    axis=-1
)

state = torch.tensor(
    state,
    dtype=torch.float32
).permute(1,0,2)


In [5]:
true_y = state

T = true_y.shape[0]

t = torch.linspace(0, T-1, T)

true_y0 = true_y[0]



In [ ]:
class Args:
    data_size = 200
    batch_time = 150
    batch_size = 20
    method = 'dopri5'
    niters = 2000
    test_freq = 20
    viz = True
    adjoint = True

args = Args()

if args.adjoint:
    from torchdiffeq import odeint_adjoint as odeint
else:
    from torchdiffeq import odeint


def get_batch():
    s = torch.from_numpy(np.random.choice(np.arange(args.data_size - args.batch_time, dtype=np.int64), args.batch_size, replace=False))
    batch_y0 = true_y[s]  # (M, D)
    batch_t = t[:args.batch_time]  # (T)
    batch_y = torch.stack([true_y[s + i] for i in range(args.batch_time)], dim=0)  # (T, M, D)
    return batch_y0, batch_t, batch_y


def makedirs(dirname):
    if not os.path.exists(dirname):
        os.makedirs(dirname)


if args.viz:
    makedirs('png')
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(12, 4), facecolor='white')
    ax_traj = fig.add_subplot(131, frameon=False)
    ax_vecfield = fig.add_subplot(132, frameon=False)
    ax_phase = fig.add_subplot(133, frameon=False)
    
    plt.show(block=False)


def visualize(true_y, pred_y, odefunc, itr):

    if args.viz:
        print(true_y.shape)
        ax_traj.cla()
        ax_traj.set_title('Trajectories')
        ax_traj.set_xlabel('t')
        ax_traj.set_ylabel('x,y')
        ax_traj.plot(t.cpu().numpy(), true_y.detach().cpu().numpy()[:, 0, 0], t.cpu().numpy(), true_y.detach().cpu().numpy()[:, 0, 1], 'g-')
        ax_traj.plot(t.cpu().numpy(), pred_y.detach().cpu().numpy()[:, 0, 0], '--', t.cpu().numpy(), pred_y.detach().cpu().numpy()[:, 0, 1], 'b--')
        ax_traj.legend()

        ax_vecfield.cla()
        ax_vecfield.set_title('Vector Field')
        ax_vecfield.set_xlabel('x')
        ax_vecfield.set_ylabel('y')
        ax_vecfield.plot(t.cpu().numpy(), true_y.detach().cpu().numpy()[:, 0, 2], t.cpu().numpy(), true_y.detach().cpu().numpy()[:, 0, 3], 'g-')
        ax_vecfield.plot(t.cpu().numpy(), pred_y.detach().cpu().numpy()[:, 0, 2], '--', t.cpu().numpy(), pred_y.detach().cpu().numpy()[:, 0, 3], 'b--')
        ax_vecfield.legend()

        ax_phase.cla()
        ax_phase.set_title('Phase Portrait')
        ax_phase.set_xlabel('x')
        ax_phase.set_ylabel('y')
        ax_phase.plot(true_y.detach().cpu().numpy()[:, 0, 0], true_y.detach().cpu().numpy()[:, 0, 1], 'g-')
        ax_phase.plot(pred_y.detach().cpu().numpy()[:, 0, 0], pred_y.detach().cpu().numpy()[:, 0, 1], 'b--')
        
        fig.tight_layout()
        save_dir = 'circle/figures/NeuralODE_rec'
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(
            os.path.join(save_dir, f'{itr:03d}.png'),
            dpi=150,
            bbox_inches='tight'
        )

        plt.pause(0.001)


class ODEFunc(nn.Module):

    def __init__(self):
        super(ODEFunc, self).__init__()
        
        '''
        АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ ДЛЯ МОДЕЛИРОВАНИЯ ВЕКТОРА ПОЛЯ
        '''
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 4)
        )


        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, val=0)

    def forward(self, t, y):
        return self.net(y)


class RunningAverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, momentum=0.99):
        self.momentum = momentum
        self.reset()

    def reset(self):
        self.val = None
        self.avg = 0

    def update(self, val):
        if self.val is None:
            self.avg = val
        else:
            self.avg = self.avg * self.momentum + val * (1 - self.momentum)
        self.val = val


if __name__ == '__main__':

    ii = 0

    func = ODEFunc()
    
    optimizer = optim.Adam(func.parameters(), lr=1e-3)
    end = time.time()

    time_meter = RunningAverageMeter(0.97)
    
    loss_meter = RunningAverageMeter(0.97)

    for itr in range(1, args.niters + 1):
        optimizer.zero_grad()

        batch_y0 = true_y[0] 
        batch_t = t             
        batch_y = true_y 

        '''
        ОБУЧЕНИЕ ПО БАТЧАМ ДЛЯ РАЗНООБРАЗИЯ ОБУЧАЮЩИХ ПРИМЕРОВ
        '''
        #batch_y0, batch_t, batch_y = get_batch()
        
        pred_y = odeint(func, batch_y0, batch_t)


        eps = 1e-8

        '''
        ВЫЧИСЛЕНИЕ FTLE ДЛЯ ОЦЕНКИ ЧУВСТВИТЕЛЬНОСТИ СИСТЕМЫ К НАЧАЛЬНЫМ УСЛОВИЯМ
        '''
        #J = torch.autograd.functional.jacobian( lambda x: odeint(func, x, batch_t)[-1], batch_y0) 
        #J2 = J.squeeze()
        #ftle = torch.log(torch.linalg.svdvals(J2).max()) / (batch_t[-1] - batch_t[0])
        ##with torch.no_grad():
        #    print(ftle.item())


        #loss_pos = torch.mean(torch.abs(pred_y[:, :, :2] - batch_y[:, :, :2]))
        #loss_vel = torch.mean(torch.abs(pred_y[:, :, 2:] - batch_y[:, :, 2:]))
        #loss = loss_pos 
       
        loss_wape_x = (
            torch.abs(pred_y[:, :, 0] - batch_y[:, :, 0]).sum()
            /
            (torch.abs(batch_y[:, :, 0]).sum() + eps)
        )

        loss_wape_y = (
            torch.abs(pred_y[:, :, 1] - batch_y[:, :, 1]).sum()
            /
            (torch.abs(batch_y[:, :, 1]).sum() + eps)
        )
        
        err_pos = torch.linalg.norm(
            pred_y[:, :, :2] - batch_y[:, :, :2],
            dim=-1
        )
        weights_pos = err_pos.detach()
        weights_pos = weights_pos / (weights_pos.mean() + eps)

        #err_vel = torch.linalg.norm(
        #    pred_y[:, :, 2:] - batch_y[:, :, 2:],
        #    dim=-1
        #)
        #weights_vel = err_vel.detach()
        #weights_vel = weights_vel / (weights_vel.mean() + eps)

        
        loss = 1000 * (weights_pos * err_pos).mean() 

        loss.backward()
        optimizer.step()

        time_meter.update(time.time() - end)
        loss_meter.update(loss.item())

        if itr % args.test_freq == 0:
            with torch.no_grad():
                pred_y = odeint(func, true_y0, t)
                loss = torch.mean(torch.abs(pred_y - true_y))
                print('Iter {:04d} | Total Loss {:.6f}'.format(itr, loss.item()))
                visualize(true_y, pred_y, func, ii)
                ii += 1

        end = time.time()
